# ⚡ Transvid: Neural Latent Media Compression Machine

Welcome to the official **Transvid** demonstration notebook!

This notebook demonstrates how Transvid encodes audio and video into **compact neural latent representations** (`.tvid` format) and reconstructs high-fidelity media back using deep neural decoders.

---

### Step 1: Verify Cloud GPU and Environment

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

### Step 2: Install Transvid Dependencies

In [ ]:
# Install dependencies directly
!pip install -q encodec torchaudio soundfile numpy zstandard tqdm

### Step 3: Generate or Upload Audio Sample
Let's create a 5-second multi-tone acoustic test signal (or upload your own `.wav`/`.mp3`).

In [ ]:
import numpy as np
import torchaudio
import torch

sample_rate = 44100
duration = 5.0
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

# Rich harmonic musical chord (C Major: C4, E4, G4)
chord = (
    0.4 * np.sin(2 * np.pi * 261.63 * t) +
    0.3 * np.sin(2 * np.pi * 329.63 * t) +
    0.3 * np.sin(2 * np.pi * 392.00 * t)
).astype(np.float32)

torchaudio.save("input_sample.wav", torch.from_numpy(chord).unsqueeze(0), sample_rate)
print("Created input_sample.wav")

### Step 4: Compress into `.tvid` Neural Latents

In [ ]:
!python -m src.cli compress input_sample.wav -o sample.tvid --bandwidth 6.0
!python -m src.cli info sample.tvid

### Step 5: Decompress `.tvid` back to Playable Audio

In [ ]:
!python -m src.cli decompress sample.tvid -o restored_sample.wav

### Step 6: Interactive Side-by-Side Listening & Size Comparison

In [ ]:
import os
from IPython.display import Audio, display, HTML

orig_size = os.path.getsize("input_sample.wav")
tvid_size = os.path.getsize("sample.tvid")
ratio = orig_size / tvid_size

display(HTML(f"""
<h3>Compression Results:</h3>
<ul>
  <li><b>Original WAV:</b> {orig_size / 1024:.1f} KB</li>
  <li><b>Transvid (.tvid):</b> {tvid_size / 1024:.1f} KB</li>
  <li><b>Savings:</b> <span style='color:green;'><b>{ratio:.1f}x smaller!</b></span> ({100 - (tvid_size/orig_size*100):.1f}% reduction)</li>
</ul>
"""))

print("Original Audio:")
display(Audio("input_sample.wav"))

print("Reconstructed Audio from .tvid:")
display(Audio("restored_sample.wav"))